# CS 430 – Assignment 2: SLCM Regression Analysis

**Course:** CS-430-01 Machine Learning  
**Target variable:** `past_due_rent`  
**Dataset:** SLCM Rent Arrears De-identified Dataset

**Prediction question:** Can household, income, employment, and housing information help predict how much past-due rent a household owes?


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Part 1 – Get to Know the Prediction Problem


### 1. Load the dataset and examine its structure.
- Load the CSV file into pandas.
- Display the first 5 rows.
- Report the number of rows and columns.
- Print the column names.
- Use `info()` to inspect the dataset.

**Answer:** Which variable will be the target? Which columns appear to be numerical? Which columns appear to be categorical?


In [ ]:
df = pd.read_csv("SLCM_rent_arrears_student_clean.csv")
print(df.head().to_string())
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nColumn names:", list(df.columns))
print()
df.info()

### Answer (Q1)

- **Target:** `past_due_rent`, the amount of rent a household owes.
- **Numerical columns:** `adults_in_household`, `children_in_household`, `monthly_household_income`, `monthly_rent`, and `past_due_rent`.
- **Categorical columns:** `household_has_65plus`, `income_sources`, `employment_status`, `has_housing_voucher`, `eviction_in_progress`, `behind_on_utilities`, `main_reason_behind_on_rent`, and `intake_quarter`.
- `record_id` is only an ID, not useful household information. The dataset has 1,754 rows and 14 columns.


### 2. Investigate missing values.
- Calculate the number of missing values in every column.
- Identify which variables contain missing data.
- Briefly describe how you plan to handle those missing values before modeling.


In [ ]:
missing = df.isna().sum()
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": (missing / len(df) * 100).round(2)})
print(missing_df.to_string())
print("Columns with missing data:", missing[missing > 0].index.tolist())
print("\nZero values (real answers, not missing):")
print("  monthly_household_income == 0:", (df["monthly_household_income"] == 0).sum())
print("  monthly_rent == 0:", (df["monthly_rent"] == 0).sum())

### Answer (Q2)

Five columns have missing values: `monthly_household_income` (108), `main_reason_behind_on_rent` (16), `monthly_rent` (8), `behind_on_utilities` (7), and `eviction_in_progress` (1). The target, `past_due_rent`, has no missing values.

I will keep the rows and fill in missing values before modeling. Missing numerical values will be filled with the training data's median. Missing categories will be filled with the most common category. These steps happen inside the pipeline, so the test data is not used to set the values. Reported values of $0 are kept because they are not missing values.


### 3. Explore the target variable: `past_due_rent`.
Calculate count, mean, median, standard deviation, minimum, maximum. Create a histogram and a boxplot.

**Answer:** What does the distribution look like? Is it symmetric or skewed? Are the mean and median similar? Do you notice any unusually high values? Why might rent-arrears data naturally contain a wide range of values?


In [ ]:
y_all = df["past_due_rent"]
stats = pd.Series({
    "count": y_all.count(), "mean": y_all.mean(), "median": y_all.median(),
    "std": y_all.std(), "min": y_all.min(), "max": y_all.max(),
})
print(stats.round(2).to_frame("past_due_rent").to_string())
print("Skewness:", round(y_all.skew(), 2))
q1, q3 = y_all.quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print(f"Boxplot upper fence: ${fence:,.0f};  rows above it: {(y_all > fence).sum()};  rows above $10,000: {(y_all > 10000).sum()}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(y_all, bins=40, edgecolor="black")
ax[0].axvline(y_all.mean(), color="red", linestyle="--", label=f"Mean ${y_all.mean():,.0f}")
ax[0].axvline(y_all.median(), color="green", linestyle="--", label=f"Median ${y_all.median():,.0f}")
ax[0].set_title("Distribution of past_due_rent"); ax[0].set_xlabel("Past-due rent ($)"); ax[0].set_ylabel("Households"); ax[0].legend()
ax[1].boxplot(y_all, vert=False)
ax[1].set_title("Boxplot of past_due_rent"); ax[1].set_xlabel("Past-due rent ($)")
plt.tight_layout(); plt.show()

### Answer (Q3)

- **Shape:** The data is right-skewed. Most households owe less, while a smaller number owe much more.
- **Mean and median:** The mean is about $2,408 and the median is $2,000. A few large amounts pull the mean upward.
- **High values:** Some households owe more than $10,000, and the largest amount is $20,000. These values are kept because they may be real.
- **Why amounts vary:** Some households may be behind by one month, while others may have gone unpaid for much longer. Rent costs and changes in income can also differ.


# Part 2 – Prepare the Data for Machine Learning


### 4. Create your input features and target.
Create `X` and `y` using `past_due_rent` as the target. Do not use `record_id` as an input feature.

**Answer:** Why should `record_id` not be used to predict past-due rent?


In [ ]:
y = df["past_due_rent"]
X = df.drop(columns=["record_id", "past_due_rent"])
print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", list(X.columns))

### Answer (Q4)

`record_id` is just a label for each row. It does not tell us about a household's finances. Using it could make the model learn a pattern based on how the data was organized instead of learning from useful information.


### 5. Separate the numerical and categorical features.
Create lists of numerical and categorical features, display both, and briefly explain why machine-learning models require different preparation for each.


In [ ]:
numerical_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = [c for c in X.columns if c not in numerical_features]
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

### Answer (Q5)

Numerical features are numbers, but they can use very different scales, so missing values are filled in and the values are scaled. Categorical features are labels or groups, so missing values are filled in and the categories are changed into numbers. They need different preparation steps because they contain different kinds of information.


### 6. Split the data into training and testing sets.
Use `test_size=0.20` and `random_state=42`. Report the dimensions of `X_train`, `X_test`, `y_train`, and `y_test`.

**Answer:** Why should the model be evaluated using data that was not used during training?


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

### Answer (Q6)

The split has 1,403 training rows and 351 testing rows. The model learns from the training data, then is tested on data it has not seen before. This helps show how well it may work on new households and makes it easier to spot overfitting. All four models use the same split for a fair comparison.


### 7. Prepare the numerical variables.
Create a numerical preprocessing pipeline that handles missing values and feature scaling (`SimpleImputer`, `StandardScaler`).

**Explain:** How did you handle missing numerical values? Why might feature scaling matter when comparing Linear, Ridge, and Lasso models?


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
print(numeric_transformer)

### Answer (Q7)

Missing income and rent values are filled with the training data's median. Then the numerical features are scaled so they are on a similar scale. Scaling is especially important for Ridge and Lasso because their penalties depend on the size of the model's coefficients. Without scaling, features measured in dollars could be treated differently just because of their units.


### 8. Prepare the categorical variables.
Create a categorical preprocessing pipeline that handles missing values and converts text categories into numbers (`SimpleImputer`, `OneHotEncoder`).

**Answer:** Why can the regression models not directly use values such as employment categories? Why is one-hot encoding appropriate? Why would assigning arbitrary numbers such as 1, 2, 3, 4 to unordered categories potentially be misleading?


In [ ]:
print("Unique values in income_sources:", X["income_sources"].nunique())
print("Categories appearing fewer than 10 times:", (X["income_sources"].value_counts() < 10).sum())

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False)),
])
print(categorical_transformer)

### Answer (Q8)

- **Why text cannot be used directly:** The regression models need numbers to make predictions.
- **Why one-hot encoding helps:** It creates a separate 0/1 column for each category, without saying that one category is greater than another.
- **Why not use 1, 2, 3, 4:** Those numbers would suggest an order that may not exist.
- **Handling rare categories:** `income_sources` has many combinations, and some appear only a few times. The encoder groups rare combinations together and ignores categories it did not see during training.


### 9. Combine your preprocessing steps.
Use a `ColumnTransformer` to combine the numerical and categorical preprocessing.

**Explain** briefly why using a preprocessing pipeline is useful.


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features),
])
print(preprocessor)

### Answer (Q9)

A pipeline keeps the cleaning and preparation steps together with the model. It makes the same steps run in the same order for training and testing, and helps prevent information from the test data leaking into training.


# Part 3 – Build and Compare Regression Models


In [ ]:
def evaluate(model, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name}:  RMSE = ${rmse:,.2f}   R² = {r2:.4f}")
    return preds, rmse, r2

baseline_rmse = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))
print(f"Baseline (predict the mean every time) RMSE = ${baseline_rmse:,.2f}")

### 10. Build a Multiple Linear Regression model.
Create a pipeline with preprocessing and Linear Regression, train it, and predict the test set. Calculate RMSE and R². Create an Actual vs. Predicted scatterplot with a diagonal reference line.

**Answer:** What does RMSE represent? What does R² tell you? How closely do the predictions match? Are there cases with large errors?


In [ ]:
linear_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", LinearRegression())])
lin_preds, lin_rmse, lin_r2 = evaluate(linear_pipe, "Linear Regression")

plt.figure(figsize=(6.5, 6))
plt.scatter(y_test, lin_preds, alpha=0.5)
lims = [0, max(y_test.max(), lin_preds.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual past_due_rent ($)"); plt.ylabel("Predicted past_due_rent ($)")
plt.title("Linear Regression: Actual vs. Predicted"); plt.legend(); plt.show()

### Answer (Q10)

- **RMSE:** The RMSE is about $1,850, so predictions are often off by a large amount. This is only a little better than always guessing the average.
- **R²:** An R² of about 0.17 means the model explains about 17% of the differences in past-due rent.
- **Graph:** The points are spread out instead of sitting close to the diagonal line, so predictions are not very accurate.
- **Large errors:** The model often predicts too low for households with very large amounts of past-due rent.


### 11. Build a Polynomial Regression model.
Use `PolynomialFeatures(degree=2, include_bias=False)` on the appropriate numerical variables, not on one-hot encoded categories. Train and calculate RMSE and R².

**Answer:** Did it perform better or worse than Linear? What does degree 2 allow? Why could a very high degree overfit? Did more complexity automatically make it better?


In [ ]:
numeric_poly_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
])
preprocessor_poly = ColumnTransformer(transformers=[
    ("num", numeric_poly_transformer, numerical_features),   # polynomial only on numerical features
    ("cat", categorical_transformer, categorical_features),  # categories left as plain one-hot
])
poly_pipe = Pipeline(steps=[("preprocess", preprocessor_poly), ("model", LinearRegression())])
poly_preds, poly_rmse, poly_r2 = evaluate(poly_pipe, "Polynomial Regression (degree 2)")
n_num_poly = poly_pipe.named_steps["preprocess"].named_transformers_["num"].named_steps["poly"].n_output_features_
print(f"Numerical features expanded from {len(numerical_features)} to {n_num_poly}")

### Answer (Q11)

- **Result:** Polynomial Regression performed a little worse than Linear Regression.
- **What degree 2 does:** It adds squared features and combinations of features, letting the model fit more complicated patterns.
- **Overfitting:** A very complex model may learn small details in the training data that do not apply to new households.
- **Conclusion:** The extra complexity did not improve the predictions in this case.


### 12. Build a Ridge Regression model.
Use the same train/test data and choose a reasonable `alpha` (you may experiment). Calculate RMSE and R².

**Answer:** How does Ridge compare with Linear? What does regularization do? Why is it useful with many features?


In [ ]:
alphas = [0.1, 1, 10, 30, 100, 300, 1000]

# Choose alpha using 5-fold cross-validation on the TRAINING data only (test set stays untouched)
ridge_search = GridSearchCV(
    Pipeline([("preprocess", preprocessor), ("model", Ridge())]),
    param_grid={"model__alpha": alphas}, cv=5, scoring="neg_root_mean_squared_error",
).fit(X_train, y_train)
cv_ridge = pd.DataFrame({"alpha": alphas, "cv_RMSE": -ridge_search.cv_results_["mean_test_score"]})
display(cv_ridge.round(2))
best_ridge_alpha = ridge_search.best_params_["model__alpha"]
print("Best Ridge alpha (by CV):", best_ridge_alpha)

ridge_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", Ridge(alpha=best_ridge_alpha))])
ridge_preds, ridge_rmse, ridge_r2 = evaluate(ridge_pipe, f"Ridge Regression (alpha={best_ridge_alpha})")

### Answer (Q12)

- **Compared with Linear:** Ridge performed slightly better, but the difference was small.
- **What regularization does:** It keeps the model's coefficients from becoming too large.
- **Why it helps:** With many features, a model can learn patterns that are just noise. Ridge can reduce that problem and help it work better on new data.


### 13. Build a Lasso Regression model.
Same training/testing data, reasonable `alpha`. Calculate RMSE and R².

**Answer:** How does Lasso compare with Linear and Ridge? What is an important difference between Ridge and Lasso? What can happen to some coefficients?


In [ ]:
lasso_alphas = [0.1, 1, 3, 10, 30, 100, 300]
lasso_search = GridSearchCV(
    Pipeline([("preprocess", preprocessor), ("model", Lasso(max_iter=20000))]),
    param_grid={"model__alpha": lasso_alphas}, cv=5, scoring="neg_root_mean_squared_error",
).fit(X_train, y_train)
cv_lasso = pd.DataFrame({"alpha": lasso_alphas, "cv_RMSE": -lasso_search.cv_results_["mean_test_score"]})
display(cv_lasso.round(2))
best_lasso_alpha = lasso_search.best_params_["model__alpha"]
print("Best Lasso alpha (by CV):", best_lasso_alpha)

lasso_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", Lasso(alpha=best_lasso_alpha, max_iter=20000))])
lasso_preds, lasso_rmse, lasso_r2 = evaluate(lasso_pipe, f"Lasso Regression (alpha={best_lasso_alpha})")

# Which coefficients did Lasso set to exactly zero?
feat_names = lasso_pipe.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(lasso_pipe.named_steps["model"].coef_, index=feat_names)
print(f"\nLasso set {(coefs == 0).sum()} of {len(coefs)} coefficients to exactly zero")
print("\nLargest remaining coefficients:")
display(coefs[coefs != 0].sort_values(key=abs, ascending=False).head(8).round(1).to_frame("coefficient"))

### Answer (Q13)

- **Comparison:** Lasso had the best results so far, although it was only a little better than Ridge and Linear Regression.
- **Ridge vs. Lasso:** Ridge makes coefficients smaller. Lasso can make some coefficients exactly zero, which removes those features from the model.
- **Result:** Lasso removed some features and kept others that were more useful for making predictions.


### 14. Compare all four models.
Create a DataFrame containing the test-set results for all four models.

**Answer:** Which has the lowest RMSE? Highest R²? Are the differences large or small? Did the most complicated model perform the best? Which model would you continue investigating, and why?


In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression", "Ridge Regression", "Lasso Regression"],
    "RMSE": [lin_rmse, poly_rmse, ridge_rmse, lasso_rmse],
    "R²": [lin_r2, poly_r2, ridge_r2, lasso_r2],
}).round({"RMSE": 2, "R²": 4})
print(results.to_string(index=False))
print("Lowest RMSE :", results.loc[results["RMSE"].idxmin(), "Model"])
print("Highest R²  :", results.loc[results["R²"].idxmax(), "Model"])

fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True, sharey=True)
for ax, (name, p) in zip(axes.ravel(), [("Linear", lin_preds), ("Polynomial", poly_preds), ("Ridge", ridge_preds), ("Lasso", lasso_preds)]):
    ax.scatter(y_test, p, alpha=0.4)
    ax.plot([0, y_test.max()], [0, y_test.max()], "r--")
    ax.set_title(name)
for ax in axes[1]: ax.set_xlabel("Actual ($)")
for ax in axes[:, 0]: ax.set_ylabel("Predicted ($)")
plt.suptitle("Actual vs. Predicted – all four models", y=1.0); plt.tight_layout(); plt.show()

### Answer (Q14)

- **Best results:** Lasso had the lowest RMSE (about $1,826) and highest R² (about 0.187).
- **Size of the difference:** The models performed similarly, with only small differences in their scores.
- **Most complex model:** Polynomial Regression was the most complex, but it performed the worst.
- **Model to continue with:** I would continue with Lasso because it had the best scores and removed some less useful features. Its advantage was small, so it would be worth testing it again with another split.


### 15. Test your selected model on two hypothetical households.
Create TWO new hypothetical cases (not copied from the dataset) that differ in several characteristics. Predict `past_due_rent` for both.

**Explain:** How are the two households different? How different are the predictions? Does the difference seem reasonable?


In [ ]:
selected_model = lasso_pipe   # selected in Question 14

hypothetical = pd.DataFrame([
    {   # Household A: small, working, has a voucher, current on utilities, lower rent
        "adults_in_household": 1, "children_in_household": 0, "household_has_65plus": "No",
        "monthly_household_income": 2800.0, "income_sources": "Full-time employment",
        "employment_status": "Employed", "has_housing_voucher": "Yes",
        "eviction_in_progress": "I don't know", "behind_on_utilities": "No",
        "main_reason_behind_on_rent": "Unexpected expense", "intake_quarter": "2026-Q3",
        "monthly_rent": 650.0,
    },
    {   # Household B: larger, lost job, no voucher, behind on utilities, higher rent
        "adults_in_household": 2, "children_in_household": 3, "household_has_65plus": "No",
        "monthly_household_income": 600.0, "income_sources": "Food benefits - TANF/SNAP",
        "employment_status": "Unemployed but want to find work", "has_housing_voucher": "No",
        "eviction_in_progress": "Yes", "behind_on_utilities": "Yes",
        "main_reason_behind_on_rent": "Lost a job/had hours cut", "intake_quarter": "2026-Q3",
        "monthly_rent": 1500.0,
    },
], index=["Household A", "Household B"])

hypothetical["predicted_past_due_rent"] = selected_model.predict(hypothetical[X.columns]).round(2)
print(hypothetical.T.to_string())
print(f"Difference (B - A): ${hypothetical['predicted_past_due_rent'].diff().iloc[-1]:,.2f}")

### Answer (Q15)

- **How they differ:** Household A has one working adult, $2,800 monthly income, a housing voucher, and $650 rent. Household B has two adults and three children, $600 monthly income, no job or voucher, and $1,500 rent. Household B is also behind on utilities and has an eviction in progress.
- **Predictions:** The model predicts about $1,160 for A and $3,230 for B.
- **Does this seem reasonable?** It makes sense that the model predicts more past-due rent for B, given the lower income and higher rent. However, predictions can be wrong by a lot, so these numbers should not be treated as exact.


# Part 4 – Interpret the Results


### 16. What did your models tell you?
Address: best model, typical prediction errors, whether Polynomial helped, whether regularization helped, what the model predicts reasonably well, where it struggles.


In [ ]:
# Evidence for Question 16: where do the selected model's errors fall?
err = pd.DataFrame({"actual": y_test.values, "predicted": lasso_preds})
err["error (pred - actual)"] = err["predicted"] - err["actual"]
bins = pd.cut(err["actual"], [-1, 1000, 3000, 6000, 20001], labels=["≤ $1,000", "$1,000–3,000", "$3,000–6,000", "> $6,000"])
print(err.groupby(bins, observed=True)["error (pred - actual)"].agg(["count", "mean"]).round(0).rename(columns={"mean": "average error ($)"}).to_string())
print(f"Mean absolute error: ${np.abs(err['error (pred - actual)']).mean():,.0f}")
print(f"Prediction range: ${err['predicted'].min():,.0f} to ${err['predicted'].max():,.0f}  |  Actual range: ${err['actual'].min():,.0f} to ${err['actual'].max():,.0f}")

### Answer (Q16)

- **Best model:** Lasso performed best, with an RMSE of about $1,826 and an R² of about 0.187.
- **Typical error:** The average absolute error was about $1,180, which is large compared with the median past-due amount of $2,000.
- **Polynomial Regression:** It performed slightly worse than Linear Regression, so adding complexity did not help.
- **Regularization:** Ridge and Lasso performed a little better than Linear Regression, but the improvement was small.
- **Where it works better:** The model does a better job with amounts closer to the middle of the data.
- **Where it struggles:** It often predicts too high for small amounts and too low for very large amounts. The available information does not explain most of the differences between households.


### 17. What is your overall finding for SLCM?
In 5–8 sentences, explain your analysis as if speaking to someone from SLCM who has not taken a machine-learning course.


### Answer (Q17)

I used household and financial information to predict how much past-due rent a household owes when it contacts SLCM. Lasso was the best model, but it only explained about 19% of the differences between households. Its predictions were often off by about $1,800, which is a large amount compared with the typical amount owed. The information collected at intake helps somewhat, but it does not explain many other reasons a household may fall behind. More detailed information, such as how long rent has gone unpaid, might help improve predictions. The model may help show general patterns across many households, but it should not be used to make decisions about individual people. These results show patterns in this dataset, not proof that any one factor causes past-due rent.
